# Post-processing a remage simulation

This tutorial takes the output of a _remage_ simulation and turns it into a
"hit" file: for every interaction in a detector we compute the quantities that
a real experiment would measure, and we write them to disk. We then build the
time-coincidence map and use it to look at the two germanium detectors together
with the surrounding liquid argon.

The whole dataset is small enough to be held in memory, which is the simplest
way to work. The last section shows what changes when it is not.

## The simulation

We use the geometry of the [remage tutorial](https://remage.readthedocs.io):
two germanium detectors, a BEGe and a coaxial one, inside a liquid argon
volume, with a $^{228}$Th source between them. The macro simulates one decay
chain per event, and keeps only the events that deposit at least 1 MeV in
germanium:

```text
/RMG/Geometry/RegisterDetector Germanium BEGe 001
/RMG/Geometry/RegisterDetector Germanium Coax 002
/RMG/Geometry/RegisterDetector Scintillator LAr 003

/run/initialize

# keep only the events that deposit at least 1 MeV in germanium
/RMG/Output/Germanium/EdepCutLow 1 MeV
/RMG/Processes/Stepping/ResetInitialDecayTime true

/RMG/Generator/Confine Volume
/RMG/Generator/Confinement/SampleOnSurface false
/RMG/Generator/Confinement/Physical/AddVolume Source

/RMG/Generator/Select GPS
/gps/particle ion
/gps/energy 0 eV
/gps/ion 90 228
/process/had/rdm/nucleusLimits 208 228 81 90

/run/beamOn 500000
```

Running it takes 38 seconds on 12 threads and writes 18 MB:

```console
$ remage --threads 12 --gdml-files geometry.gdml --output-file th228.lh5 \
         --merge-output-files -- th228.mac
```

_remage_ groups the energy depositions ("steps") of each event into "hits", one
per detector, and writes one table per detector. This is the input we start
from.

In [ ]:
stp_file = "th228.lh5"
gdml_file = "geometry.gdml"
hit_file = "th228_hit.lh5"

## The detector geometry

The processors that model the germanium detectors need to know their shape and
where they sit. Both come from the same GDML file that _remage_ simulated, read
back with [pyg4ometry](https://pyg4ometry.readthedocs.io). The detector
metadata stored in it by
[legend-pygeom-tools](https://legend-pygeom-tools.readthedocs.io) is turned
into an HPGe object by
[legend-pygeom-hpges](https://legend-pygeom-hpges.readthedocs.io).

In [ ]:
import pyg4ometry
from pygeomhpges import make_hpge
from pygeomtools.detectors import get_sensvol_metadata

registry = pyg4ometry.gdml.Reader(gdml_file).getRegistry()

hpges = {
    name: make_hpge(get_sensvol_metadata(registry, name), registry=None)
    for name in ("BEGe", "Coax")
}
positions = {name: registry.physicalVolumeDict[name].position.eval() for name in hpges}

{name: f"{hpge.mass:.0f} g" for name, hpge in hpges.items()}

## Reading the data

We read the detector table as an LGDO {class}`~lgdo.types.table.Table` and view
it as an [awkward](https://awkward-array.org) array. The awkward view is what
the processors work on: one entry per hit, and inside it a list of steps. The
`with_units` flag attaches the physical units of each field, which the
processors read and convert as needed.

In [ ]:
import awkward as ak
import lh5

stp = lh5.read("stp/Coax", stp_file)
data = stp.view_as("ak", with_units=True)

print(f"{len(data)} hits, {ak.sum(ak.num(data.edep, axis=-1))} steps")
data.fields

In [ ]:
data[0].show(limit_cols=200)

## Applying processors

A "processor" computes one new quantity per hit. The first one we apply is the
correction for the inactive layer at the surface of a germanium detector: a
charge deposited close to the n+ electrode is only partly collected, so the
measured energy is lower than the deposited one.

`distance_to_surface` gives the distance of every step to the surface, and
`piecewise_linear_activeness` turns it into the fraction of charge that is
collected there. Summing the weighted energies over the steps of a hit gives
the energy the detector would report.

In [ ]:
from reboost.hpge import surface
from reboost.math import functions

distance = surface.distance_to_surface(
    data.xloc, data.yloc, data.zloc, hpges["Coax"], positions["Coax"]
)

# 1.5 mm of inactive layer, the outermost 0.3 mm of which is fully dead
activeness = functions.piecewise_linear_activeness(distance, fccd_in_mm=1.5, dlf=0.2)

deposited = ak.sum(data.edep, axis=-1)
collected = ak.sum(data.edep * activeness, axis=-1)

In [ ]:
import hist
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 12})

fig, ax = plt.subplots(figsize=(12, 4))
for values, label in ((deposited, "deposited"), (collected, "collected")):
    h = hist.new.Reg(125, 1000, 3500, name="energy [keV]").Double().fill(values)
    h.plot(ax=ax, yerr=False, flow="none", label=label)
ax.set_yscale("log")
ax.set_xlim(1000, 3500)
ax.set_ylabel("counts / 20 keV")
_ = ax.legend()

The correction moves counts out of the full-energy peaks into the continuum
below them. The hits that happen entirely in the inactive layer collect almost
no charge and end up far below the range plotted here.

The simulation has no electronic noise either, so the peaks are single bins
wide. `gaussian_sample` smears each energy with the detector resolution.

In [ ]:
from reboost.math import stats

energy = stats.gaussian_sample(collected, sigma=1.5, seed=1234)  # unit is keV

The last quantity we compute is `r90`, the radius of the sphere around the
energy-weighted centre of a hit that contains 90% of its energy. It is small
for an interaction concentrated in one spot and large for a hit spread over
the detector, so it separates single-site from multi-site events, the way pulse
shape discrimination does in the real experiment.

In [ ]:
from reboost.hpge import psd

r90 = psd.r90(data.edep, data.xloc, data.yloc, data.zloc)

## Writing the hit file

`init_hit_table` starts an output table from the input one. It carries over the
two fields that identify a hit in time, the Geant4 event identifier `evtid` and
the time of the hit `t0`, and nothing else. We add the quantities we computed
and write the table with `write_hit_table_chunk`.

The `uid` is the number the detector was registered with in the macro. Passing
it makes the function link the table under `hit/__by_uid__/det002`, the same
layout _remage_ uses, so that the tools that look detectors up by identifier
also work on our file.

In [ ]:
from pathlib import Path

from lgdo import Array

from reboost import io

hit = io.init_hit_table(stp)
hit.add_field("energy", Array(energy, attrs={"units": "keV"}))
hit.add_field("r90", Array(r90, attrs={"units": "mm"}))

# the function appends, so remove the output of an earlier run
Path(hit_file).unlink(missing_ok=True)

io.write_hit_table_chunk(hit, "hit/Coax", hit_file, uid=2)

The other two detectors go the same way: the same processors for the BEGe, and
for the argon only the summed energy, since it has neither a surface effect nor
a pulse shape to model. `get_remage_detector_uids` reads the identifiers of all
of them from the input file.

In [ ]:
from reboost import utils

uids = utils.get_remage_detector_uids(stp_file)

# start from a clean file, the writer appends to what it finds
Path(hit_file).unlink(missing_ok=True)

for uid, name in uids.items():
    stp = lh5.read(f"stp/{name}", stp_file)
    data = stp.view_as("ak", with_units=True)
    out = io.init_hit_table(stp)

    if name in hpges:
        distance = surface.distance_to_surface(
            data.xloc, data.yloc, data.zloc, hpges[name], positions[name]
        )
        activeness = functions.piecewise_linear_activeness(distance, fccd_in_mm=1.5, dlf=0.2)
        collected = ak.sum(data.edep * activeness, axis=-1)
        out.add_field(
            "energy",
            Array(
                stats.gaussian_sample(collected, sigma=1.5, seed=1234),
                attrs={"units": "keV"},
            ),
        )
        out.add_field(
            "r90",
            Array(
                psd.r90(data.edep, data.xloc, data.yloc, data.zloc),
                attrs={"units": "mm"},
            ),
        )
    else:
        out.add_field("energy", Array(ak.sum(data.edep, axis=-1), attrs={"units": "keV"}))

    io.write_hit_table_chunk(out, f"hit/{name}", hit_file, uid=uid)

lh5.show(hit_file)

The germanium tables and the argon table have different fields, which is fine:
each detector table stands on its own.

## Event analysis

Everything so far worked on one detector at a time. A processor sees the hits of
a single table and returns a new column for it, and nothing in it depends on
what the other detectors were doing. This is where the detector models live: the
surface response, the energy resolution, the pulse shape.

An experiment, though, does not record detectors, it records events: what all
the detectors saw at the same time. Going from the one to the other is the
second half of the post-processing, and it starts from the time-coincidence map
(TCM). Each row of the TCM is an event, and lists the hits, across all
detectors, that happened in the same Geant4 event and close enough in time to be
read out together.

_remage_ writes one next to the steps, and it describes our file just as well:
the processors did not change the number of rows nor their order, so row `i` of
`hit/Coax` is the same hit as row `i` of `stp/Coax`.

In [ ]:
tcm = lh5.read_as("tcm", stp_file, "ak")
tcm[:5].show(limit_cols=200)

> **When to rebuild it.** The map points at rows, so it only holds as long as
> those rows do. Rebuild it with `reboost.tcm.build_remage_tcm` if you drop or
> reorder hits, if you put several files together, or if you want a coincidence
> window other than the one _remage_ used. It writes the map into the file you
> give it, next to the detector tables:
>
> ```python
> reboost.tcm.build_remage_tcm(hit_file, hit_file, coin_window_in_ns=1000)
> ```

`table_key` is the detector identifier and `row_in_table` the row of that
detector's table, so the two together point at one hit. To read a field of
every hit of an event, `read_hit_field_by_tcm` follows those pointers and
returns the values with the same jagged structure as the TCM.

In [ ]:
energies = io.read_hit_field_by_tcm(tcm, hit_file, "energy")

# keep the events in which at least one germanium detector fired
event = tcm[ak.any(tcm.table_key != 3, axis=-1)]
energies = energies[ak.any(tcm.table_key != 3, axis=-1)]

# total energy in germanium and in argon, per event
germanium = ak.sum(energies[event.table_key != 3], axis=-1)
argon = ak.sum(energies[event.table_key == 3], axis=-1)

# how many germanium detectors fired in each event
multiplicity = ak.num(energies[event.table_key != 3], axis=-1)
multiplicity.show(limit_rows=1)

Most events fire a single germanium detector, and one in eight fires both:
those are the events in which a gamma scattered from one detector into the
other.

The argon says something about the rest of the decay. Rejecting the events that
deposited energy in it leaves only those in which everything that the decay
released stayed inside a germanium detector.

In [ ]:
veto = argon < 100  # unit is keV

fig, ax = plt.subplots(figsize=(12, 4))
for values, label in ((germanium, "all events"), (germanium[veto], "no energy in argon")):
    h = hist.new.Reg(125, 1000, 3500, name="energy [keV]").Double().fill(values)
    h.plot(ax=ax, yerr=False, flow="none", label=label)
ax.set_yscale("log")
ax.set_xlim(1000, 3500)
ax.set_ylabel("counts / 20 keV")
_ = ax.legend()

Only 13% of the events survive. The source sits in the middle of the argon, so
almost every decay leaves energy there. The 2615 keV line of $^{208}$Tl is cut
down as hard as the continuum, to 7%, because that gamma is emitted in cascade
with a 583 keV one: even when the 2615 keV gamma is fully absorbed in
germanium, its partner is usually stopped by the argon. The small bump at
3198 keV is the case where it is not: both gammas of the cascade absorbed in
germanium, which is what leaves the argon empty. A real experiment uses the
veto the other way around, to reject the background events that a source
deliberately produces here.

## Scaling up

Everything above held the whole file in memory. A production simulation is
larger than the memory of the machine, and then the file has to be read in
pieces. `LH5Iterator` returns one chunk of rows at a time, and
`write_hit_table_chunk` appends each result to the output file, so the loop
body is the same code as before.

In [ ]:
chunked_file = "th228_hit_chunked.lh5"
Path(chunked_file).unlink(missing_ok=True)

for stp in lh5.LH5Iterator(stp_file, "stp/Coax", buffer_len=5000):
    data = stp.view_as("ak", with_units=True)
    out = io.init_hit_table(stp)

    distance = surface.distance_to_surface(
        data.xloc, data.yloc, data.zloc, hpges["Coax"], positions["Coax"]
    )
    activeness = functions.piecewise_linear_activeness(distance, fccd_in_mm=1.5, dlf=0.2)
    out.add_field(
        "energy",
        Array(ak.sum(data.edep * activeness, axis=-1), attrs={"units": "keV"}),
    )

    io.write_hit_table_chunk(out, "hit/Coax", chunked_file, uid=2)

lh5.read_as("hit/Coax", chunked_file, "ak").energy

Two things to keep in mind. `write_hit_table_chunk` appends to whatever it
finds, so the output file has to be deleted before the first chunk, as above.
And the TCM has to be built after the loop, over the finished file, because a
coincidence can span two chunks.

A last note for the case where the work is split across several jobs. Chunks of
rows do not respect event boundaries: the hits of one event can fall in two
different chunks, and a job that reads a fixed number of rows can cut an event
in half. `reboost.io.get_rows_in_event_range` gives the first row and the
number of rows of a detector table that belong to a range of events, ready to
be passed to `LH5Iterator` as `i_start` and `n_entries`, so that each job reads
whole events. It needs the detector tables to be stored in event order, which
is the case for a simulation run on a single thread.